# Sunspot Forecasting — Experiments

This notebook explores forecast horizon sensitivity:
1. **30-Day Horizon** — does the hybrid model compete with ARIMA at monthly granularity?
2. **Horizon Sweep (1–30 days)** — where does the hybrid add the most value over a naive baseline?

**Prerequisites:** Run `00-EDA.ipynb` first (downloads data), then `01-Analysis_and_Modeling.ipynb` (exports `models/`).

In [ ]:
import sys
import os
import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.append(os.path.abspath('..'))
    os.chdir('..')

os.makedirs('reports', exist_ok=True)

from src.utils import load_config
from src.data import load_data
from src.features import create_features, prepare_target
from src.train import train_evaluate, expanding_walk_forward_splits

config = load_config('config.yaml')
df = load_data(config['data']['url'], save_path=config['data']['save_path'])

start_period = '1965-01-01'
df_filtered = df.loc[start_period:].copy()
print(f"Period: {df_filtered.index.min().date()} → {df_filtered.index.max().date()}")
print(f"Total days: {len(df_filtered):,}")

### Helper — compute monthly baselines (McNish-Lincoln & ARIMA)

Used across both experiments. Run once here so we don't repeat it below.

In [ ]:
monthly = df_filtered['SUNSPOTS'].resample('MS').mean().dropna()

INIT_MONTHS  = 132   # 11-year warm-up
ARIMA_WINDOW = 120   # sliding 10-year window keeps ARIMA fast

mcnish_rmse, mcnish_mae = [], []
arima_rmse,  arima_mae  = [], []

for i in range(INIT_MONTHS, len(monthly) - 1):
    train  = monthly.iloc[:i]
    actual = monthly.iloc[i]

    mcnish_pred = train.iloc[-13:].mean()
    mcnish_rmse.append((actual - mcnish_pred) ** 2)
    mcnish_mae.append(abs(actual - mcnish_pred))

    try:
        fit = ARIMA(train.iloc[-ARIMA_WINDOW:], order=(5, 1, 0)).fit()
        arima_pred = max(fit.forecast(steps=1).iloc[0], 0)
        arima_rmse.append((actual - arima_pred) ** 2)
        arima_mae.append(abs(actual - arima_pred))
    except Exception:
        pass

mcn_r = np.sqrt(np.mean(mcnish_rmse))
mcn_m = np.mean(mcnish_mae)
arm_r = np.sqrt(np.mean(arima_rmse))
arm_m = np.mean(arima_mae)

print(f"McNish-Lincoln — RMSE: {mcn_r:.2f}  MAE: {mcn_m:.2f}")
print(f"ARIMA(5,1,0)   — RMSE: {arm_r:.2f}  MAE: {arm_m:.2f}")

---
## Experiment 1 — 30-Day Horizon

The main model targets a **5-day forecast**. Here we ask: can the same hybrid architecture compete with ARIMA at a **30-day horizon**?

The lag-based feature set is designed for short autocorrelation windows, so performance at 30 days is expected to degrade — this experiment quantifies by how much.

In [ ]:
config_30d = copy.deepcopy(config)
config_30d['features']['target_shift'] = -30
config_30d['evaluation']['val_size']   = 30

df_feat_30d = create_features(df_filtered, config_30d['features']['lags'], config_30d['features']['rolling_windows'])
df_feat_30d = prepare_target(df_feat_30d, shift=config_30d['features']['target_shift'])
X_30d = df_feat_30d.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y_30d = df_feat_30d['target']

print("Training hybrid model at 30-day horizon...")
results_30d = train_evaluate(X_30d, y_30d, config_30d)
print(f"Hybrid 30d — RMSE: {results_30d['hybrid_rmse']:.4f}  MAE: {results_30d['hybrid_mae']:.4f}")

# Daily baselines at 30-day horizon
naive30_rmse, naive30_mae = [], []
roll30_rmse,  roll30_mae  = [], []

for Xtr, ytr, Xval, yval in expanding_walk_forward_splits(
    X_30d, y_30d,
    config_30d['evaluation']['initial_train_size'],
    config_30d['evaluation']['val_size'],
    config_30d['evaluation']['step_size']
):
    if len(Xtr) < 365:
        continue
    yval_linear = np.expm1(yval)
    naive_pred  = np.full(len(yval), np.expm1(ytr.iloc[-1]))
    roll_pred   = np.full(len(yval), np.expm1(ytr.iloc[-30:].mean()))

    naive30_rmse.append(np.sqrt(mean_squared_error(yval_linear, naive_pred)))
    naive30_mae.append(mean_absolute_error(yval_linear, naive_pred))
    roll30_rmse.append(np.sqrt(mean_squared_error(yval_linear, roll_pred)))
    roll30_mae.append(mean_absolute_error(yval_linear, roll_pred))

hyb30_r, hyb30_m = results_30d['hybrid_rmse'], results_30d['hybrid_mae']
n30_r,   n30_m   = np.mean(naive30_rmse),      np.mean(naive30_mae)
r30_r,   r30_m   = np.mean(roll30_rmse),        np.mean(roll30_mae)

def delta(baseline, model):
    pct = (baseline - model) / baseline * 100
    return f"{pct:+.1f}%"

print()
print(f"{'Model':<28} {'Granularity':<12} {'RMSE':>8} {'MAE':>8}")
print('─' * 60)
print(f"{'Naive (lag-1)':<28} {'daily':<12} {n30_r:>8.2f} {n30_m:>8.2f}")
print(f"{'Rolling mean (30d)':<28} {'daily':<12} {r30_r:>8.2f} {r30_m:>8.2f}")
print(f"{'McNish-Lincoln (13m smooth)':<28} {'monthly':<12} {mcn_r:>8.2f} {mcn_m:>8.2f}")
print(f"{'ARIMA(5,1,0)':<28} {'monthly':<12} {arm_r:>8.2f} {arm_m:>8.2f}")
print(f"{'Hybrid (ours, 30d horizon)':<28} {'daily':<12} {hyb30_r:>8.2f} {hyb30_m:>8.2f}")
print()
print(f"vs Naive (lag-1)   — RMSE: {delta(n30_r, hyb30_r)}  MAE: {delta(n30_m, hyb30_m)}")
print(f"vs Roll mean (30d) — RMSE: {delta(r30_r, hyb30_r)}  MAE: {delta(r30_m, hyb30_m)}")
print(f"vs ARIMA(5,1,0)    — RMSE: {delta(arm_r, hyb30_r)}  MAE: {delta(arm_m, hyb30_m)}")
print()
print("Note: positive = hybrid wins, negative = baseline wins.")

**Takeaway:** At 30 days the lag-based feature set loses its edge — autocorrelation signal decays and ARIMA's explicit AR structure becomes more effective. The hybrid's sweet spot is ≤5 days ahead.

---
## Experiment 2 — Horizon Sweep (1–30 days)

We train and evaluate the hybrid model across horizons `[1, 2, 3, 5, 7, 10, 14, 21, 30]` to map where the model adds the most value over the naive baseline.

Two things to watch:
- **Absolute RMSE**: always increases with horizon (harder to predict further out)
- **% improvement over naive**: the sweet spot — where our model gains the most relative to simply saying "tomorrow = today"

In [ ]:
horizons = [1, 2, 3, 5, 7, 10, 14, 21, 30]
sweep = []  # (horizon, hybrid_rmse, hybrid_mae, naive_rmse, naive_mae)

for h in horizons:
    print(f"  horizon={h:>2}d ...", end=" ", flush=True)

    cfg_h = copy.deepcopy(config)
    cfg_h['features']['target_shift'] = -h
    cfg_h['evaluation']['val_size']   = h

    df_h = create_features(df_filtered, cfg_h['features']['lags'], cfg_h['features']['rolling_windows'])
    df_h = prepare_target(df_h, shift=cfg_h['features']['target_shift'])
    X_h  = df_h.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
    y_h  = df_h['target']

    res_h = train_evaluate(X_h, y_h, cfg_h)

    naive_rmse_h, naive_mae_h = [], []
    for Xtr, ytr, Xval, yval in expanding_walk_forward_splits(
        X_h, y_h,
        cfg_h['evaluation']['initial_train_size'],
        cfg_h['evaluation']['val_size'],
        cfg_h['evaluation']['step_size']
    ):
        if len(Xtr) < 365:
            continue
        yval_linear = np.expm1(yval)
        naive_pred  = np.full(len(yval), np.expm1(ytr.iloc[-1]))
        naive_rmse_h.append(np.sqrt(mean_squared_error(yval_linear, naive_pred)))
        naive_mae_h.append(mean_absolute_error(yval_linear, naive_pred))

    sweep.append((
        h,
        res_h['hybrid_rmse'], res_h['hybrid_mae'],
        np.mean(naive_rmse_h), np.mean(naive_mae_h)
    ))
    print(f"hybrid RMSE={res_h['hybrid_rmse']:.2f}  naive RMSE={np.mean(naive_rmse_h):.2f}")

In [ ]:
print(f"{'Horizon':<10} {'Hybrid RMSE':>12} {'Naive RMSE':>12} {'Improv. RMSE':>14} {'Improv. MAE':>12}")
print('─' * 64)
for h, h_r, h_m, n_r, n_m in sweep:
    imp_r = (n_r - h_r) / n_r * 100
    imp_m = (n_m - h_m) / n_m * 100
    print(f"{h:<10} {h_r:>12.2f} {n_r:>12.2f} {imp_r:>13.1f}% {imp_m:>11.1f}%")

hs     = [r[0] for r in sweep]
h_rmse = [r[1] for r in sweep]
n_rmse = [r[3] for r in sweep]
imp_r  = [(r[3]-r[1])/r[3]*100 for r in sweep]
imp_m  = [(r[4]-r[2])/r[4]*100 for r in sweep]
best_h = hs[int(np.argmax(imp_r))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hs, n_rmse, marker='o', label='Naive (lag-1)', color='gray',    linestyle='--')
ax1.plot(hs, h_rmse, marker='o', label='Hybrid (ours)', color='#d62728')
ax1.set_title('Absolute RMSE vs Forecast Horizon')
ax1.set_xlabel('Horizon (days)')
ax1.set_ylabel('RMSE')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(hs, imp_r, marker='o', color='steelblue',  label='RMSE improvement')
ax2.plot(hs, imp_m, marker='s', color='darkorange', label='MAE improvement', linestyle='--')
ax2.axvline(best_h, color='red', linestyle=':', alpha=0.7, label=f'Peak at {best_h}d')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('% Improvement over Naive vs Forecast Horizon')
ax2.set_xlabel('Horizon (days)')
ax2.set_ylabel('Improvement over naive (%)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
fig.savefig('reports/07_horizon_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPeak RMSE improvement at horizon = {best_h} days")